In [0]:
# Dynamic path resolution (no hardcoded user email)
_user = spark.sql("SELECT current_user()").first()[0]
_base = f"file:/Workspace/Users/{_user}/databricks_ai/MachineLearning"

df = spark.read.format("csv").option("header", "true").load(f"{_base}/diabetes.csv")
display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
   
data = df.dropna().select(col("Pregnancies").astype("int"),
                           col("Glucose").astype("int"),
                          col("BloodPressure").astype("int"),
                          col("SkinThickness").astype("int"),
                          col("Insulin").astype("int"),
                          col("BMI").astype("float"),
                          col("DiabetesPedigreeFunction").astype("float"),
                          col("Age").astype("int"),
                          col("Outcome").astype("int")
                          )
display(data)

In [0]:
splits = data.randomSplit([0.7, 0.3])
train = splits[0]
test = splits[1]
print ("Training Rows:", train.count(), " Testing Rows:", test.count())

# Performing Feature Engineering

## Normalizing/scaling our features

In [0]:

from pyspark.ml.feature import VectorAssembler, MinMaxScaler

numericFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction"]
numericColVector = VectorAssembler(inputCols=numericFeatures, outputCol = "numericFeatures")
vectorizedData = numericColVector.transform(train)

minMax = MinMaxScaler(inputCol= numericColVector.getOutputCol(), outputCol="normalizedFeatures")
scaledData = minMax.fit(vectorizedData).transform(vectorizedData)

compareNumerics = scaledData.select("numericFeatures", "normalizedFeatures")
display(compareNumerics)

## Preparing the features and the labels

In [0]:
preppedData = scaledData[col("normalizedFeatures").alias("features"), col("Outcome").alias("label")]
display(preppedData)

## Train a Machine Learning Model

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10, regParam=0.3)
model = lr.fit(preppedData)
print ("Model trained!")
     

## Testing the prepared model

In [0]:
vectorizedTestData = numericColVector.transform(test)
scaledTestData = minMax.fit(vectorizedTestData).transform(vectorizedTestData)
preppedTestData = scaledTestData[col("normalizedFeatures").alias("features"), col("Outcome").alias("label")]
   
# Get predictions
prediction = model.transform(preppedTestData)
predicted = prediction.select("features", "probability", col("prediction").astype("Int"), col("label").alias("trueLabel"))
display(predicted)

## Evaluating Our Model

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
   
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
   
# Simple accuracy
accuracy = evaluator.evaluate(prediction, {evaluator.metricName:"accuracy"})
print("Accuracy:", accuracy)
   
# Individual class metrics
labels = [0,1]
print("\nIndividual class metrics:")
for label in sorted(labels):
    print ("Class %s" % (label))
   
    # Precision
    precision = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                                evaluator.metricName:"precisionByLabel"})
    print("\tPrecision:", precision)
   
    # Recall
    recall = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                             evaluator.metricName:"recallByLabel"})
    print("\tRecall:", recall)
   
    # F1 score
    f1 = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                         evaluator.metricName:"fMeasureByLabel"})
    print("\tF1 Score:", f1)
   
# Weighted (overall) metrics
overallPrecision = evaluator.evaluate(prediction, {evaluator.metricName:"weightedPrecision"})
print("Overall Precision:", overallPrecision)
overallRecall = evaluator.evaluate(prediction, {evaluator.metricName:"weightedRecall"})
print("Overall Recall:", overallRecall)
overallF1 = evaluator.evaluate(prediction, {evaluator.metricName:"weightedFMeasure"})
print("Overall F1 Score:", overallF1)

## Using a Pipeline for Encapsulation

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml.classification import LogisticRegression
   

numFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
   
# Define the feature engineering and model training algorithm steps
numVector = VectorAssembler(inputCols=numFeatures, outputCol="numericFeatures")
numScaler = MinMaxScaler(inputCol = numVector.getOutputCol(), outputCol="normalizedFeatures")
featureVector = VectorAssembler(inputCols=["normalizedFeatures"], outputCol="Features")
algo = LogisticRegression(labelCol="Outcome", featuresCol="Features", maxIter=10, regParam=0.3)
   
# Chain the steps as stages in a pipeline
pipeline = Pipeline(stages=[ numVector, numScaler, featureVector, algo])
   
# Use the pipeline to prepare data and fit the model algorithm
model = pipeline.fit(train)
print ("Model trained!")

In [0]:
prediction = model.transform(test)
predicted = prediction.select("Features", "probability", col("prediction").astype("Int"), col("Outcome").alias("trueLabel"))
display(predicted)

## Save Model for Online Inference

Steps:
1. Log the trained pipeline with MLflow (includes signature + input example)
2. Register in workspace model registry
3. Validate locally
4. Deploy to a Model Serving endpoint (REST API)

In [0]:
import mlflow
from mlflow.models import infer_signature

# Use workspace model registry (legacy cluster without UC)
mlflow.set_registry_uri("databricks")

# Prepare a sample input for signature inference
sample_input = test.select("Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
                           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age").limit(5).toPandas()

# Get predictions on sample for output signature
sample_prediction = model.transform(test).select("prediction").limit(5).toPandas()

signature = infer_signature(sample_input, sample_prediction)

# Log the pipeline model
with mlflow.start_run(run_name="diabetes_logistic_regression") as run:
    mlflow.log_params({"maxIter": 10, "regParam": 0.3})
    mlflow.log_metric("accuracy", accuracy)

    model_info = mlflow.spark.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input,
        registered_model_name="diabetes_classifier",
    )

print(f"Model logged: {model_info.model_uri}")
print(f"Run ID: {run.info.run_id}")

In [0]:
# Validate the model loads and predicts correctly before deploying
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

# Test with sample data
result = loaded.predict(sample_input)
print("Predictions from loaded model:")
print(result)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)
from databricks.sdk.errors import ResourceAlreadyExists, ResourceConflict
import time

w = WorkspaceClient()

endpoint_name = "diabetes-classifier"
config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name="diabetes_classifier",
            entity_version="1",
            scale_to_zero_enabled=True,
            workload_size="Small",
        )
    ]
)

try:
    w.serving_endpoints.create_and_wait(name=endpoint_name, config=config)
    print(f"\u2705 Endpoint '{endpoint_name}' created!")
except ResourceAlreadyExists:
    # Endpoint exists — wait for it to be ready
    print(f"Endpoint '{endpoint_name}' already exists. Waiting for it to be ready...")
    endpoint = w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint_name)
    print(f"\u2705 Endpoint '{endpoint_name}' is ready! State: {endpoint.state.ready}")
except ResourceConflict:
    # Endpoint is mid-update — wait for it to finish
    print(f"Endpoint '{endpoint_name}' is currently updating. Waiting...")
    endpoint = w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint_name)
    print(f"\u2705 Endpoint '{endpoint_name}' is ready! State: {endpoint.state.ready}")

host = w.config.host.rstrip("/")
if not host.startswith("https://"):
    host = f"https://{host}"
print(f"URL: {host}/serving-endpoints/{endpoint_name}/invocations")

In [0]:
import requests
import json

# Query the endpoint with sample data
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")

url = f"https://{host}/serving-endpoints/diabetes-classifier/invocations"

payload = {
    "dataframe_records": [
        {
            "Pregnancies": 6, "Glucose": 148, "BloodPressure": 72,
            "SkinThickness": 35, "Insulin": 0, "BMI": 33.6,
            "DiabetesPedigreeFunction": 0.627, "Age": 50
        }
    ]
}

response = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json=payload,
)

print("Status:", response.status_code)
print("Prediction:", json.dumps(response.json(), indent=2))